# Task Archiving & Worker Offboarding Workflow

This notebook automates the process of:
1. Querying the credentials layer to find workers flagged for deletion (`delete_today = YES`)
2. Finding completed tasks assigned to each flagged worker
3. Copying those tasks to an archive layer
4. Deleting them from the source layer
5. Removing each flagged mobile worker's account

**How to use:** Run cells top to bottom. Each section is self-contained with clear inputs and outputs.

## 1. Imports & Setup

The [ArcGIS Python API](https://developers.arcgis.com/python/) gives us two key entry points:
- **`GIS`** — authenticates and connects to your ArcGIS Online org. Almost every API call flows through this object.
- **`FeatureLayer`** — wraps a hosted feature service endpoint, unlocking methods to query, edit, and delete features without writing raw REST calls.

We also add basic logging so you can see what the script is doing at each step — especially useful when processing large batches.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
import logging
import tempfile
import os

# from getpass import getpass <--- Uncomment if you want to use username/password authentication

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

gis = GIS("home")

# username = input("Enter ArcGIS username: ")
# password = getpass("Enter ArcGIS password: ")
# gis = GIS("https://www.arcgis.com", username, password)

## 2. Configuration

**Only edit this cell.** Set your layer URLs, the assignee username, and the status code for completed tasks.

In [ ]:
CONFIG = {
    # --- Layer URLs ---
    "task_layer_url": "https://services.arcgis.com/123/arcgis/rest/services/Task_Layer/FeatureServer/0",
    "archived_task_layer_url": "https://services.arcgis.com/k123/arcgis/rest/services/Archived_Task_Layer/FeatureServer/0",
    "credentials_url": "https://services.arcgis.com/123/arcgis/rest/services/Credentials/FeatureServer/0",
    # --- Workers ---
    # Workers flagged with delete_today = YES in the credentials layer are picked up automatically.
    "reassign_to": None,
    # --- Filter values ---
    "completed_status": "3",  # Status code for 'completed' tasks in esritask_status
    # --- Performance ---
    "batch_size": 1000,  # Number of features to add/delete per API call
}

## 3. Helper Functions

Each helper below is its own cell so you can read, test, and run them one at a time.
Run all of them before moving to Section 4 — the execution cells depend on every function being loaded.

### Helper 1 of 7 — `build_where_clause`

Turns a status code into a SQL filter string that the feature layer can understand.

| | |
|---|---|
| **Input** | `status` (string) — the numeric code for the task status you want to match, e.g. `'3'` for completed |
| **Returns** | A SQL WHERE clause string, e.g. `"esritask_status = '3'"`, ready to pass straight into a layer query |

In [ ]:
def build_where_clause(status: str) -> str:
    """
    Builds a SQL WHERE clause to filter tasks by their status code.

    A WHERE clause is just a filter condition — think of it like the
    "where" part of a sentence: "give me all rows WHERE status equals 3".
    This function formats that sentence correctly so the ArcGIS API
    can understand it.

    Input:
        status (str):
            The task status code to filter by.
            Example: "3" means completed tasks in this dataset.

    Returns:
        str:
            A SQL WHERE clause string.
            Example output: "esritask_status = '3'"
            Pass this directly to FeatureLayer.query(where=...).
    """
    where = f"esritask_status = '{status}'"
    logger.info(f"WHERE clause: {where}")
    return where

### Helper 2 of 7 — `query_matching_features`

Searches a feature layer and returns the matching records plus their IDs.

| | |
|---|---|
| **Input** | `task_layer` (FeatureLayer) — the layer to search |
| **Input** | `where_clause` (string) — the SQL filter from `build_where_clause` |
| **Returns** | A tuple of two things: (1) a FeatureSet containing the full records with geometry, and (2) a plain list of integer Object IDs. If nothing matches, returns `(None, [])` |

In [ ]:
def query_matching_features(task_layer: FeatureLayer, where_clause: str) -> tuple:
    """
    Searches a feature layer and returns matching records plus their Object IDs.

    Uses a two-step approach to keep things efficient:
      Step 1 — Ask the server for Object IDs only (very fast, no geometry).
               If nothing comes back, stop early and return empty results.
      Step 2 — If IDs were found, fetch the full records including geometry.
    This avoids downloading large payloads when there is nothing to process.

    Inputs:
        task_layer (FeatureLayer):
            The ArcGIS feature layer you want to search.
            Example: FeatureLayer(CONFIG["task_layer_url"], gis)

        where_clause (str):
            A SQL filter string that narrows down which records to return.
            Example: "esritask_status = '3'"
            Build this with build_where_clause() above.

    Returns:
        tuple: (feature_set, object_ids)

        feature_set (FeatureSet or None):
            The full matching records, including attributes and geometry.
            This is what you pass into copy_to_archive() and copy_attachments().
            Returns None if no records matched.

        object_ids (list of int):
            A plain list of integer IDs for the matched records.
            Example: [12, 45, 103]
            You will pass these into delete_from_source() later.
            Returns an empty list [] if no records matched.
    """
    oid_result = task_layer.query(where=where_clause, return_ids_only=True)
    object_ids = oid_result.get("objectIds", [])

    if not object_ids:
        logger.info("No features found matching the query.")
        return None, []

    logger.info(
        f"Found {len(object_ids)} matching feature(s). Fetching full records..."
    )
    feature_set = task_layer.query(where=where_clause, return_geometry=True)
    return feature_set, object_ids

### Helper 3 of 7 — `copy_to_archive`

Writes a set of features into the archive layer and returns the new IDs created there.

| | |
|---|---|
| **Input** | `archived_task_layer` (FeatureLayer) — the archive layer to write into |
| **Input** | `feature_set` (FeatureSet) — the records returned by `query_matching_features` |
| **Input** | `fields` (list of strings, optional) — if provided, only these field names are copied; copies everything if left as `None` |
| **Input** | `batch_size` (int, optional) — how many records to send per API call; defaults to `1000` |
| **Returns** | A list of integer Object IDs that were just created in the archive layer. Compare the length of this list against your source Object IDs before deleting anything |

In [ ]:
def copy_to_archive(
    archived_task_layer: FeatureLayer,
    feature_set,
    fields: list = None,
    batch_size: int = 1000,
) -> list:
    """
    Copies features from a FeatureSet into the archive layer.

    Records are sent in batches (default 1000 at a time) to avoid hitting
    the API's payload size limits when dealing with large datasets.

    An optional fields list lets you copy only specific columns — useful
    when the source and archive layers have different schemas.

    Inputs:
        archived_task_layer (FeatureLayer):
            The archive layer you want to write records into.
            Example: FeatureLayer(CONFIG["archived_task_layer_url"], gis)

        feature_set (FeatureSet):
            The records to copy, returned by query_matching_features().

        fields (list of str, optional):
            Names of the specific fields you want to carry over.
            Example: ["task_name", "assigned_to", "completed_date"]
            Leave as None to copy all fields.

        batch_size (int, optional):
            How many records to send to the API in a single call.
            Default is 1000. Lower this if you see timeout errors.

    Returns:
        list of int:
            The Object IDs that were created in the archive layer.
            Example: [201, 202, 203]
            If the length of this list matches your source object_ids list,
            the copy was successful and it is safe to delete the originals.
    """
    if not feature_set or not feature_set.features:
        logger.info("No features to copy.")
        return []

    fields_set = set(fields) if fields is not None else None

    new_features = [
        {
            "attributes": (
                {f: feature.attributes.get(f) for f in fields_set}
                if fields_set is not None
                else dict(feature.attributes)
            ),
            **(
                {"geometry": feature.geometry}
                if hasattr(feature, "geometry") and feature.geometry
                else {}
            ),
        }
        for feature in feature_set.features
    ]

    added_oids = [
        r.get("objectId")
        for i in range(0, len(new_features), batch_size)
        for r in archived_task_layer.edit_features(
            adds=new_features[i : i + batch_size]
        ).get("addResults", [])
        if r.get("success") or not logger.error(f"Add failed: {r.get('error')}")
    ]

    logger.info(
        f"Successfully copied {len(added_oids)}/{len(new_features)} feature(s) to archive."
    )
    return added_oids

### Helper 4 of 7 — `copy_attachments`

Moves photos, PDFs, and other files attached to source records over to their matching archive records.

| | |
|---|---|
| **Input** | `task_layer` (FeatureLayer) — the layer to download attachments from |
| **Input** | `archived_task_layer` (FeatureLayer) — the archive layer to upload attachments into |
| **Input** | `feature_set` (FeatureSet) — the original records (used to read their source Object IDs) |
| **Input** | `archived_oids` (list of int) — the new Object IDs in the archive, returned by `copy_to_archive`; must be in the same order as the source records |
| **Returns** | A dictionary with two keys: `'copied'` (int, how many attachments succeeded) and `'failed'` (int, how many could not be transferred) |

In [ ]:
def copy_attachments(
    task_layer: FeatureLayer,
    archived_task_layer: FeatureLayer,
    feature_set,
    archived_oids: list,
) -> dict:
    """
    Downloads every file attachment from the source records and re-uploads
    each one to the matching record in the archive layer.

    Source and archive records are matched by position: the first record in
    feature_set corresponds to the first ID in archived_oids, the second to
    the second, and so on. This positional pairing is guaranteed because
    copy_to_archive adds records in the same order they appear in feature_set.

    Each attachment is saved to a temporary file during the transfer and
    deleted immediately afterwards to avoid filling up disk space.

    Inputs:
        task_layer (FeatureLayer):
            The layer whose attachments you want to copy from.

        archived_task_layer (FeatureLayer):
            The archive layer to copy attachments into.

        feature_set (FeatureSet):
            The original records returned by query_matching_features().
            Used here only to read the source Object IDs in order.

        archived_oids (list of int):
            The Object IDs created in the archive layer by copy_to_archive().
            Must be in the same order as the records in feature_set.
            Example: [201, 202, 203]

    Returns:
        dict:
            A summary of what happened.
            Example: {"copied": 5, "failed": 1}
            "copied" — number of attachments successfully transferred.
            "failed" — number of attachments that could not be transferred
                       (errors are logged above this summary).
    """
    copied = 0
    failed = 0

    task_layer_oids = [
        f.attributes.get("objectid") or f.attributes.get("OBJECTID")
        for f in feature_set.features
    ]
    oid_pairs = list(zip(task_layer_oids, archived_oids))

    for src_oid, arch_oid in oid_pairs:
        attachments = task_layer.attachments.get_list(oid=src_oid)

        if not attachments:
            logger.info(f"  OID {src_oid}: no attachments, skipping.")
            continue

        logger.info(
            f"  OID {src_oid}: copying {len(attachments)} attachment(s) to archive OID {arch_oid}"
        )

        for att in attachments:
            att_id = att["id"]
            att_name = att["name"]
            tmp_path = None
            try:
                tmp_dir = tempfile.mkdtemp()
                tmp_path = task_layer.attachments.download(
                    oid=src_oid,
                    attachment_id=att_id,
                    save_path=tmp_dir,
                )
                if isinstance(tmp_path, list):
                    tmp_path = tmp_path[0]

                result = archived_task_layer.attachments.add(
                    oid=arch_oid, file_path=tmp_path
                )

                if result.get("addAttachmentResult", {}).get("success"):
                    logger.info(f"    '{att_name}' copied successfully.")
                    copied += 1
                else:
                    logger.error(f"    '{att_name}' add failed: {result}")
                    failed += 1

            except Exception as e:
                logger.error(f"    '{att_name}' error: {e}")
                failed += 1

            finally:
                if tmp_path and os.path.exists(tmp_path):
                    os.remove(tmp_path)

    logger.info(f"Attachment copy complete — {copied} succeeded, {failed} failed.")
    return {"copied": copied, "failed": failed}

### Helper 5 of 7 — `delete_from_source`

Permanently removes records from the source layer using their Object IDs.

| | |
|---|---|
| **Input** | `task_layer` (FeatureLayer) — the layer to delete from |
| **Input** | `object_ids` (list of int) — the IDs to delete, returned earlier by `query_matching_features` |
| **Input** | `batch_size` (int, optional) — how many records to delete per API call; defaults to `1000` |
| **Returns** | An integer: the total count of records that were successfully deleted |

> **Important:** Only call this after `copy_to_archive` has confirmed all records landed safely in the archive.

In [ ]:
def delete_from_source(
    task_layer: FeatureLayer,
    object_ids: list,
    batch_size: int = 1000,
) -> int:
    """
    Permanently deletes records from the source layer by their Object IDs.

    Deletions are sent in batches (default 1000 at a time) so this works
    reliably even when removing thousands of records at once.

    Only call this function after you have verified that copy_to_archive()
    returned the expected number of IDs. Deleting before confirming the
    copy succeeded will result in data loss.

    Inputs:
        task_layer (FeatureLayer):
            The layer you want to delete records from.
            Example: FeatureLayer(CONFIG["task_layer_url"], gis)

        object_ids (list of int):
            The Object IDs of the records to delete.
            These come from query_matching_features().
            Example: [12, 45, 103]

        batch_size (int, optional):
            How many records to delete per API call.
            Default is 1000. Lower this if you see timeout errors.

    Returns:
        int:
            The number of records that were successfully deleted.
            Example: 3  (meaning all three IDs above were removed)
            If this is less than len(object_ids), check the logs for errors.
    """
    if not object_ids:
        logger.info("No Object IDs provided — nothing to delete.")
        return 0

    deleted_count = 0
    for i in range(0, len(object_ids), batch_size):
        batch = object_ids[i : i + batch_size]
        result = task_layer.edit_features(deletes=batch)

        for r in result.get("deleteResults", []):
            if r.get("success"):
                deleted_count += 1
            else:
                logger.error(f"Delete failed: {r.get('error')}")

    logger.info(f"Deleted {deleted_count}/{len(object_ids)} feature(s) from source.")
    return deleted_count

### Helper 6 of 7 — `delete_mobile_worker`

Removes a single ArcGIS Online user account.

| | |
|---|---|
| **Input** | `gis` (GIS) — your authenticated ArcGIS connection (requires admin privileges) |
| **Input** | `username` (string) — the exact ArcGIS Online username to delete |
| **Input** | `reassign_to` (string, optional) — if the user owns content or groups, provide another username here to transfer ownership before deletion; leave as `None` only if the account owns nothing |
| **Returns** | `True` if the account was deleted successfully, `False` if it was not found or the deletion failed |

In [ ]:
def delete_mobile_worker(
    gis: GIS,
    username: str,
    reassign_to: str = None,
) -> bool:
    """
    Deletes a mobile worker account from ArcGIS Online.

    If the account owns any maps, layers, or groups, the deletion will fail
    unless you provide a reassign_to username. That username will receive
    all of the deleted user's content before the account is removed.

    Inputs:
        gis (GIS):
            Your authenticated ArcGIS Online connection.
            Must be logged in with an administrator account.

        username (str):
            The ArcGIS Online username of the account to delete.
            Example: "johndoe_dev26"

        reassign_to (str, optional):
            The username of another account to transfer content to
            before deleting. Set to None only if you are certain the
            account being deleted owns no content or groups.
            Example: "jane_admin"

    Returns:
        bool:
            True  — the account was found and deleted successfully.
            False — the account was not found, or the deletion failed.
    """
    user = gis.users.get(username)

    if not user:
        print(f"User '{username}' not found.")
        return False

    result = user.delete(reassign_to=reassign_to)

    if result:
        print(f"User '{username}' successfully deleted.")
    else:
        print(f"Failed to delete user '{username}'.")

    return result

### Helper 7 of 7 — `get_workers_to_delete`

Reads the credentials feature layer and collects every username where `delete_today` is set to `YES`.

| | |
|---|---|
| **Input** | `gis` (GIS) — your authenticated ArcGIS connection |
| **Input** | `credentials_url` (string) — the REST endpoint URL for the credentials feature layer |
| **Returns** | A Python `set` of username strings whose `delete_today` field equals `YES`. A set is used so duplicate rows in the layer never cause the same worker to be processed twice. Returns an empty set if no one is flagged |

In [ ]:
def get_workers_to_delete(gis: GIS, credentials_url: str) -> set:
    """
    Queries the credentials feature layer and returns the usernames of
    every worker whose delete_today field is set to YES.

    The WHERE clause filter runs on the server, so only the flagged rows
    are sent back over the network — efficient even for large tables.

    Usernames are stored in a set, which automatically removes duplicates.
    This means if the same worker appears in the layer more than once with
    delete_today = YES, they are still only processed a single time.

    Workers with delete_today = NO (or any value other than YES) are
    logged and skipped entirely.

    Inputs:
        gis (GIS):
            Your authenticated ArcGIS Online connection.

        credentials_url (str):
            The REST endpoint URL for the credentials feature layer.
            Set this in CONFIG["credentials_url"] above.

    Returns:
        set of str:
            Usernames flagged for deletion.
            Example: {"johndoe_dev26", "janesmith_field3"}
            Returns an empty set if no workers are flagged today.
    """
    cred_layer = FeatureLayer(credentials_url, gis)

    result = cred_layer.query(
        where="UPPER(delete_today) = 'YES'",
        out_fields="username,delete_today",
        return_geometry=False,
    )

    usernames_to_delete = set()

    for feature in result.features:
        username = feature.attributes.get("username")
        delete_flag = (feature.attributes.get("delete_today") or "").strip().upper()

        if delete_flag == "YES" and username:
            usernames_to_delete.add(username)
            logger.info(f"Flagged for deletion: {username}")
        else:
            logger.info(
                f"Skipping '{username}' — delete_today = {feature.attributes.get('delete_today')!r}"
            )

    logger.info(f"Workers queued for deletion: {usernames_to_delete}")
    return usernames_to_delete

## 4. Resolve Workers Flagged for Deletion

Query the credentials layer for any record where `delete_today = YES`.
The results are stored in a **set** (`workers_to_delete`) so duplicates are automatically removed.
Each username in the set will go through the full archive -> delete -> offboard pipeline below.

In [ ]:
workers_to_delete = get_workers_to_delete(gis, CONFIG["credentials_url"])

if not workers_to_delete:
    print("No workers flagged for deletion today. Exiting.")
else:
    print(f"Workers to delete ({len(workers_to_delete)}): {workers_to_delete}")

## 5. Archive & Delete Tasks, then Offboard Each Worker

Iterates over every username in `workers_to_delete`.
For each worker:
1. Queries their completed tasks from the source layer
2. Copies those tasks (and attachments) to the archive layer
3. Deletes the originals from the source
4. Removes their ArcGIS Online account

A per-worker summary is printed at the end.

In [ ]:
task_layer = FeatureLayer(CONFIG["task_layer_url"], gis)
archived_task_layer = FeatureLayer(CONFIG["archived_task_layer_url"], gis)

summary = []  # Collect per-worker results

for username in workers_to_delete:
    print(f"\n{"="*60}")
    print(f"Processing worker: {username}")
    print(f"{"="*60}")

    # ── Step 1: Query completed tasks ───────────────────────────
    where_clause = build_where_clause(CONFIG["completed_status"])
    feature_set, object_ids = query_matching_features(task_layer, where_clause)
    print(f"  Tasks to archive: {len(object_ids)}")

    archived_oids = []
    attachment_summary = {"copied": 0, "failed": 0}

    if object_ids:
        # ── Step 2: Copy features to archive ────────────────────
        archived_oids = copy_to_archive(
            archived_task_layer,
            feature_set,
            batch_size=CONFIG["batch_size"],
        )
        print(f"  Archived: {len(archived_oids)}/{len(object_ids)} feature(s)")

        if len(archived_oids) != len(object_ids):
            print(
                f"  Count mismatch for {username} — skipping attachment copy and source delete."
            )
            summary.append(
                {
                    "username": username,
                    "status": "SKIPPED — archive mismatch",
                    "tasks_archived": len(archived_oids),
                    "tasks_deleted": 0,
                }
            )
            continue

        # ── Step 3: Copy attachments ─────────────────────────────
        attachment_summary = copy_attachments(
            task_layer=task_layer,
            archived_task_layer=archived_task_layer,
            feature_set=feature_set,
            archived_oids=archived_oids,
        )
        print(
            f"  Attachments — copied: {attachment_summary['copied']}, failed: {attachment_summary['failed']}"
        )

        # ── Step 4: Delete originals from source ─────────────────
        deleted_count = delete_from_source(
            task_layer, object_ids, batch_size=CONFIG["batch_size"]
        )
        print(f"  Deleted {deleted_count}/{len(object_ids)} feature(s) from source")
    else:
        print(
            f"  No completed tasks found for {username} — skipping archive/delete steps."
        )
        deleted_count = 0

    # ── Step 5: Offboard the worker ──────────────────────────────
    offboard_ok = delete_mobile_worker(
        gis=gis,
        username=username,
        reassign_to=CONFIG["reassign_to"],
    )

    summary.append(
        {
            "username": username,
            "status": "✅ deleted" if offboard_ok else "❌ offboard failed",
            "tasks_archived": len(archived_oids),
            "tasks_deleted": deleted_count,
            "attachments": attachment_summary,
        }
    )

# ── Final summary ────────────────────────────────────────────────
print(f"\n{"="*60}")
print("OFFBOARDING SUMMARY")
print(f"{"="*60}")
for row in summary:
    print(
        f"  {row['username']:30s} {row['status']}  "
        f"(archived: {row['tasks_archived']}, deleted: {row['tasks_deleted']})"
    )